In [43]:
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os
import json
import gc

OUTPUT_DIR = Path("../Output")

In [28]:
dotenv_path = find_dotenv()
print("dotenv path:", dotenv_path)

load_dotenv(dotenv_path, override=True)
API_KEY = os.getenv("OPEN_API_KEY")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-4.1-mini"

IMAGE_DATA_DIR = Path("../Dataset/dataset_image")
IMAGE_DATA_DIR = Path(IMAGE_DATA_DIR)

dotenv path: d:\Enrichment Real\sarcasm-detection\.env


In [29]:
train_metadata = pd.read_csv(OUTPUT_DIR / "train_metadata.csv")
train_embt = np.load(OUTPUT_DIR / "train_embt.npy")
train_embv = np.load(OUTPUT_DIR / "train_embv.npy")

print("train_metadata:", train_metadata.shape)
print("train_embt:", train_embt.shape)
print("train_embv:", train_embv.shape)

train_metadata: (19816, 4)
train_embt: (19816, 1, 768)
train_embv: (19816, 1, 768)


In [30]:
metadata_df = train_metadata.copy().reset_index(drop=True)


if "row_id" not in metadata_df.columns:
    metadata_df["row_id"] = np.arange(len(metadata_df))

metadata_df["row_id"] = metadata_df["row_id"].astype(int)


if "image_id" not in metadata_df.columns:
    metadata_df["image_id"] = (
        metadata_df["image_path"]
        .astype(str)
        .str.extract(r'([^\\/]+)\.[^.\\/]+$')[0]
    )

metadata_df["image_id"] = metadata_df["image_id"].astype(str)
metadata_df["label"] = metadata_df["label"].astype(int)

print("metadata_df:", metadata_df.shape)
display(metadata_df.head())

metadata_df: (19816, 5)


,image_path,text,label,image_id,row_id
0,..\Dataset\dataset_image\840006160660983809.jpg,<user> thanks for showing up for our appointme...,1,840006160660983809,0
1,..\Dataset\dataset_image\908913372199915520.jpg,haha .,1,908913372199915520,1
2,..\Dataset\dataset_image\916496521406726145.jpg,i love waiting <num> min for a cab - such shor...,1,916496521406726145,2
3,..\Dataset\dataset_image\916364004129304576.jpg,22 super funny quotes <user>,1,916364004129304576,3
4,..\Dataset\dataset_image\853866052589154304.jpg,goog morning,1,853866052589154304,4


In [31]:
metadata_df = train_metadata.copy().reset_index(drop=True)

metadata_df["row_id"] = np.arange(len(metadata_df))

if "image_id" not in metadata_df.columns:
    metadata_df["image_id"] = (
        metadata_df["image_path"]
        .astype(str)
        .str.extract(r'([^\\/]+)\.[^.\\/]+$')[0]
    )

metadata_df["image_id"] = metadata_df["image_id"].astype(str)
metadata_df["label"] = metadata_df["label"].astype(int)

display(metadata_df.head())

,image_path,text,label,image_id,row_id
0,..\Dataset\dataset_image\840006160660983809.jpg,<user> thanks for showing up for our appointme...,1,840006160660983809,0
1,..\Dataset\dataset_image\908913372199915520.jpg,haha .,1,908913372199915520,1
2,..\Dataset\dataset_image\916496521406726145.jpg,i love waiting <num> min for a cab - such shor...,1,916496521406726145,2
3,..\Dataset\dataset_image\916364004129304576.jpg,22 super funny quotes <user>,1,916364004129304576,3
4,..\Dataset\dataset_image\853866052589154304.jpg,goog morning,1,853866052589154304,4


In [32]:
def l2_normalize_np(x, axis=-1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)


def fuse_embeddings_np(embv, embt, alpha=0.5, eps=1e-12):
    embv = np.asarray(embv, dtype=np.float32)
    embt = np.asarray(embt, dtype=np.float32)

    fused = alpha * embv + (1 - alpha) * embt
    fused = l2_normalize_np(fused, axis=-1, eps=eps)

    return fused.astype(np.float32)

In [33]:
ALPHA = 0.5

train_fused_embedding = fuse_embeddings_np(
    train_embv,
    train_embt,
    alpha=ALPHA
)

if train_fused_embedding.ndim == 3 and train_fused_embedding.shape[1] == 1:
    train_fused_embedding = train_fused_embedding.squeeze(1)

train_fused_embedding = np.ascontiguousarray(train_fused_embedding, dtype=np.float32)

print("metadata_df:", metadata_df.shape)
print("train_fused_embedding:", train_fused_embedding.shape)

assert len(metadata_df) == train_fused_embedding.shape[0]

metadata_df: (19816, 5)
train_fused_embedding: (19816, 768)


In [34]:
result_df = pd.read_csv(OUTPUT_DIR / "few-shot-main-result.csv")

print(result_df.shape)
display(result_df.head())

(2000, 14)


,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores,error
0,4181,819692441481711617,..\Dataset\dataset_image\819692441481711617.jpg,i 'm rethinking everything now,0,"The phrase ""I'm rethinking everything now"" pai...",The post expresses a thoughtful and sincere re...,"['914592921814433794', '921202878571864065']","['you ever think about that ? no , you only th...","[0.744874119758606, 0.7275177240371704]","['819694428277379074', '818607947706200064']","[""i 'm rethinking everything now"", 'wayment !']","[1.0000001192092896, 0.9519281983375549]",NaN
1,5418,923000568997539840,..\Dataset\dataset_image\923000568997539840.jpg,# truth about,1,The post uses exaggeration and ironic phrasing...,The text presents a straightforward sentiment ...,"['712358912264110080', '694584307655012358']","['true .', '# truth']","[0.7744077444076538, 0.7734344601631165]","['822225305779785730', '819694201831096321']","['exactly !', 'but really though']","[0.7239524722099304, 0.7229138016700745]",NaN
2,1850,818238963156680705,..\Dataset\dataset_image\818238963156680705.jpg,took a nap and slept through a pretty sunset i...,0,The post sarcastically contrasts the disappoin...,The post expresses a straightforward sentiment...,"['839491693049171968', '698338846287785984']","['haha when you know people care ! .... jk', ""...","[0.6213423013687134, 0.5821914672851562]","['822227785137655809', '822585733273808899']",['first thing i see on snap is food <user> <us...,"[0.7123841643333435, 0.6349201202392578]",NaN
3,5650,877881532404379648,..\Dataset\dataset_image\877881532404379648.jpg,- akhileshyadav is struggling for a govt job r...,1,The post sarcastically suggests that Akhilesh ...,The post straightforwardly describes Akhilesh ...,"['880436067270356993', '877715414763130880']",[': a govt employee got delighted after rescui...,"[0.7288748025894165, 0.7151439189910889]","['819332300399779841', '818238095594291200']",['queen of flowers queen of farmers priyanka g...,"[0.5446099638938904, 0.5340462923049927]",NaN
4,9656,822594021897961472,..\Dataset\dataset_image\822594021897961472.jpg,quand jma fait signer des cracks > > > > >,0,The post text suggests a boastful claim about ...,The post expresses pride and confidence in hav...,"['686926157288202241', '902261356774207489']",['im in the process of sealing a £ 12m switch ...,"[0.5847525596618652, 0.5751825571060181]","['819688464740450304', '822587074209583104']","['i have officially completed life .', 'brace ...","[0.6404160261154175, 0.6285971999168396]",NaN


In [37]:
output_path = OUTPUT_DIR / "few-shot-main-result.csv"

In [36]:
failed_mask = (
    result_df["sarcasm_rationale"].isna() |
    result_df["not_sarcasm_rationale"].isna() |
    (result_df["sarcasm_rationale"].astype(str).str.strip() == "") |
    (result_df["not_sarcasm_rationale"].astype(str).str.strip() == "")
)

failed_indices = result_df.index[failed_mask].tolist()

print("Failed rows:", len(failed_indices))
display(result_df.loc[failed_indices])

Failed rows: 4


,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores,error
1022,9952,827653272055844864,..\Dataset\dataset_image\827653272055844864.jpg,nice prayerbreakfast <user> ! u certainly have...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unable to allocate 58.1 MiB for an array with ...
1023,2511,695029150239870976,..\Dataset\dataset_image\695029150239870976.jpg,the west african black rhino has been official...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unable to allocate 58.1 MiB for an array with ...
1024,17389,722501204740321281,..\Dataset\dataset_image\722501204740321281.jpg,great to see <user> is focusing on the importa...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unable to allocate 58.1 MiB for an array with ...
1025,5426,929330917025468416,..\Dataset\dataset_image\929330917025468416.jpg,wow ... such a high standard ... clearly trump...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unable to allocate 58.1 MiB for an array with ...


In [38]:
import time
import base64
import mimetypes
import numpy as np
import pandas as pd
from pathlib import Path
from pydantic import BaseModel, Field

class SingleRationaleOutput(BaseModel):
    rationale: str = Field(
        description="A concise rationale from the requested stance only."
    )


SYSTEM_PROMPT = """
You are a careful annotator for multimodal sarcasm reasoning.

You are given:
1. A query post consisting of text and an image.
2. Retrieved examples that already have labels, such as sarcasm or non-sarcasm.
3. A requested standpoint, either sarcastic or non-sarcastic.

The query label is unknown and must not be assumed.
The retrieved example labels are known and should be used only as supporting references for comparison.

Generate a rationale from the requested standpoint only.

Do not make a final classification decision.
Do not predict the query label.
Do not mention a gold label for the query.
Do not say whether the requested standpoint is correct or incorrect.
""".strip()


def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\n", " ").strip()


def label_name(label):
    return "sarcasm" if int(label) == 1 else "non-sarcasm"


def get_image_path(row):
    if "image_id" in row and pd.notna(row["image_id"]):
        return IMAGE_DATA_DIR / f'{str(row["image_id"])}.jpg'

    raw_path = str(row["image_path"]).replace("\\", "/")
    filename = Path(raw_path).name
    return IMAGE_DATA_DIR / filename


def image_to_data_url(image_path):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(str(image_path))

    if mime_type is None:
        suffix = image_path.suffix.lower()
        if suffix in [".jpg", ".jpeg"]:
            mime_type = "image/jpeg"
        elif suffix == ".png":
            mime_type = "image/png"
        elif suffix == ".webp":
            mime_type = "image/webp"
        else:
            mime_type = "image/jpeg"

    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{b64}"

In [39]:
def to_numpy_query(query_embedding):
    if hasattr(query_embedding, "detach"):
        query_vec = query_embedding.squeeze(0).detach().cpu().numpy()
    else:
        query_vec = np.asarray(query_embedding).squeeze()

    return query_vec.astype("float32", copy=False)


def _topk_from_mask(scores, mask, metadata_df, top_k):
    candidate_idx = np.flatnonzero(mask)

    if len(candidate_idx) == 0:
        return metadata_df.iloc[[]].copy()

    k = min(top_k, len(candidate_idx))
    candidate_scores = scores[candidate_idx]

    local_top = np.argpartition(candidate_scores, -k)[-k:]
    local_top = local_top[np.argsort(candidate_scores[local_top])[::-1]]

    top_indices = candidate_idx[local_top]

    results = metadata_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]

    return results


def retrieve_top_k_by_label(
    query_embedding,
    db_embeddings,
    metadata_df,
    top_k_each=2,
    sarcasm_label=1,
    non_sarcasm_label=0,
    exclude_row_id=None,
    exclude_image_id=None
):
    query_vec = to_numpy_query(query_embedding)

    db_embeddings = np.asarray(db_embeddings, dtype=np.float32)
    scores = db_embeddings @ query_vec

    if exclude_row_id is not None and "row_id" in metadata_df.columns:
        row_mask = metadata_df["row_id"].astype(int).to_numpy() == int(exclude_row_id)
        scores[row_mask] = -np.inf

    if exclude_image_id is not None and "image_id" in metadata_df.columns:
        image_mask = metadata_df["image_id"].astype(str).to_numpy() == str(exclude_image_id)
        scores[image_mask] = -np.inf

    labels = metadata_df["label"].astype(int).to_numpy()

    sarcasm_res = _topk_from_mask(
        scores=scores,
        mask=(labels == int(sarcasm_label)),
        metadata_df=metadata_df,
        top_k=top_k_each
    )

    non_sarcasm_res = _topk_from_mask(
        scores=scores,
        mask=(labels == int(non_sarcasm_label)),
        metadata_df=metadata_df,
        top_k=top_k_each
    )

    return sarcasm_res, non_sarcasm_res

In [40]:
def format_retrieved_examples(df, title):
    lines = [f"{title}:"]

    if df is None or len(df) == 0:
        lines.append("- No retrieved examples available.")
        return "\n".join(lines)

    for i, (_, r) in enumerate(df.iterrows(), start=1):
        lines.append(
            f"{i}. Text: {safe_text(r.get('text', ''))}\n"
            f"   Example label: {label_name(r.get('label', 0))}\n"
            f"   Similarity score: {float(r.get('score', 0.0)):.4f}"
        )

    return "\n".join(lines)


def build_stance_prompt(query_row, sarcasm_res, non_sarcasm_res, stance):
    query_text = safe_text(query_row["text"])

    prompt = f"""
Given a query post that consists of text and an image, please give a rationale of why the query post may be {stance}.

Query text:
{query_text}

Retrieved labeled examples for comparison:

{format_retrieved_examples(sarcasm_res, "Top 2 retrieved examples labeled sarcasm")}

{format_retrieved_examples(non_sarcasm_res, "Top 2 retrieved examples labeled non-sarcasm")}

Task:
Write only one rationale from the standpoint that the query post may be {stance}.

Rules:
- The query label is not provided.
- The retrieved example labels are provided and may be used as comparison evidence.
- Do not make a final classification decision.
- Do not predict the query label.
- Do not mention the true/gold label of the query.
- Use the query text and visible image context as the main evidence.
- Use retrieved labeled examples only as supporting references.
- Do not simply copy the retrieved examples.
- Keep the rationale concise, around 1-3 sentences.
""".strip()

    return prompt


def generate_single_stance_rationale(
    row,
    sarcasm_res,
    non_sarcasm_res,
    stance,
    model=MODEL_NAME,
    include_image=True,
    image_detail="low"
):
    prompt = build_stance_prompt(
        query_row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance=stance
    )

    content = [
        {
            "type": "input_text",
            "text": prompt
        }
    ]

    if include_image:
        image_path = get_image_path(row)
        content.append(
            {
                "type": "input_image",
                "image_url": image_to_data_url(image_path),
                "detail": image_detail
            }
        )

    response = client.responses.parse(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=[
            {
                "role": "user",
                "content": content
            }
        ],
        text_format=SingleRationaleOutput,
    )

    return response.output_parsed.rationale

In [42]:
def generate_rationale_for_row(
    row,
    metadata_df,
    db_embeddings,
    top_k_each=2,
    model=MODEL_NAME,
    include_image=True,
    image_detail="low",
    sleep_s=0.0
):
    row_id = int(row["row_id"])

    query_embedding = db_embeddings[row_id].reshape(1, -1)

    sarcasm_res, non_sarcasm_res = retrieve_top_k_by_label(
        query_embedding=query_embedding,
        db_embeddings=db_embeddings,
        metadata_df=metadata_df,
        top_k_each=top_k_each,
        sarcasm_label=1,
        non_sarcasm_label=0,
        exclude_row_id=row_id,
        exclude_image_id=row["image_id"]
    )

    sarcasm_rationale = generate_single_stance_rationale(
        row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance="sarcastic",
        model=model,
        include_image=include_image,
        image_detail=image_detail
    )

    if sleep_s > 0:
        time.sleep(sleep_s)

    not_sarcasm_rationale = generate_single_stance_rationale(
        row=row,
        sarcasm_res=sarcasm_res,
        non_sarcasm_res=non_sarcasm_res,
        stance="non-sarcastic",
        model=model,
        include_image=include_image,
        image_detail=image_detail
    )

    if sleep_s > 0:
        time.sleep(sleep_s)

    return {
        "row_id": row_id,
        "image_id": str(row["image_id"]),
        "image_path": row.get("image_path", None),
        "text": row["text"],
        "true_label": int(row["label"]),

        "sarcasm_rationale": sarcasm_rationale,
        "not_sarcasm_rationale": not_sarcasm_rationale,

        "top_sarcasm_ids": sarcasm_res["image_id"].astype(str).tolist(),
        "top_sarcasm_texts": sarcasm_res["text"].astype(str).tolist(),
        "top_sarcasm_scores": sarcasm_res["score"].astype(float).tolist(),

        "top_non_sarcasm_ids": non_sarcasm_res["image_id"].astype(str).tolist(),
        "top_non_sarcasm_texts": non_sarcasm_res["text"].astype(str).tolist(),
        "top_non_sarcasm_scores": non_sarcasm_res["score"].astype(float).tolist(),
    }

In [44]:
def to_csv_cell(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value


for n, idx in enumerate(failed_indices, start=1):
    try:
        row_id = int(result_df.loc[idx, "row_id"])

        row_match = metadata_df[metadata_df["row_id"].astype(int) == row_id]

        if len(row_match) == 0:
            raise ValueError(f"row_id {row_id} not found in metadata_df")

        row = row_match.iloc[0]

        fixed_result = generate_rationale_for_row(
            row=row,
            metadata_df=metadata_df,
            db_embeddings=train_fused_embedding,
            top_k_each=2,
            model=MODEL_NAME,
            include_image=True,
            image_detail="low",
            sleep_s=0.1
        )

        for col, val in fixed_result.items():
            result_df.at[idx, col] = to_csv_cell(val)

        if "error" in result_df.columns:
            result_df.at[idx, "error"] = ""

        print(
            f"[FIXED] {n}/{len(failed_indices)} "
            f"idx={idx} row_id={row_id} image_id={fixed_result['image_id']}"
        )

    except Exception as e:
        print(
            f"[FAILED AGAIN] {n}/{len(failed_indices)} "
            f"idx={idx} row_id={result_df.loc[idx, 'row_id']}"
        )
        print(e)

        if "error" in result_df.columns:
            result_df.at[idx, "error"] = str(e)

    result_df.to_csv(output_path, index=False)
    gc.collect()

print("Done. Updated file saved to:", output_path)

C:\Users\WILLIAM S L\AppData\Local\Temp\ipykernel_9520\3360781759.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '827653272055844864' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[idx, col] = to_csv_cell(val)


[FIXED] 1/4 idx=1022 row_id=9952 image_id=827653272055844864
[FIXED] 2/4 idx=1023 row_id=2511 image_id=695029150239870976
[FIXED] 3/4 idx=1024 row_id=17389 image_id=722501204740321281
[FIXED] 4/4 idx=1025 row_id=5426 image_id=929330917025468416
Done. Updated file saved to: ..\Output\few-shot-main-result.csv
